# PPO: Donkey Kong (Default Hyperparameters)

**Algorithm:** PPO  
**Environment:** Donkey Kong (`ALE/DonkeyKong-v5`)  
**Learning rate:** default (3e-4)  
**Parallel envs:** 8  

In [6]:
# Configuration
ALGO            = "ppo"
ENV_KEY         = "donkeykong"
TOTAL_TIMESTEPS = 2_000_000
CHECKPOINT_FREQ = 100_000
LEARNING_RATE   = None
RUN_NAME        = None

In [2]:
import os

import ale_py
import gymnasium as gym
from stable_baselines3 import DQN, PPO
from stable_baselines3.common.callbacks import CheckpointCallback
from stable_baselines3.common.env_util import make_atari_env
from stable_baselines3.common.vec_env import VecFrameStack

gym.register_envs(ale_py)

ENV_IDS = {
    "qbert":      "QbertNoFrameskip-v4",
    "donkeykong": "ALE/DonkeyKong-v5",
}

run_id = f"{ALGO}_{ENV_KEY}" + (f"_{RUN_NAME}" if RUN_NAME else "")
checkpoint_dir = os.path.join("checkpoints", run_id)
os.makedirs(checkpoint_dir, exist_ok=True)
os.makedirs("logs", exist_ok=True)

print(f"Run ID      : {run_id}")
print(f"Checkpoints : {checkpoint_dir}")

Run ID      : ppo_donkeykong
Checkpoints : checkpoints/ppo_donkeykong


In [3]:
n_envs = 8 if ALGO == "ppo" else 1
vec_env = make_atari_env(ENV_IDS[ENV_KEY], n_envs=n_envs, seed=0)
vec_env = VecFrameStack(vec_env, n_stack=4)
print(f"Environment : {ENV_IDS[ENV_KEY]}  |  n_envs: {n_envs}")

A.L.E: Arcade Learning Environment (version 0.11.2+ecc1138)
[Powered by Stella]


Environment : ALE/DonkeyKong-v5  |  n_envs: 8


## Fresh Training

Checkpoints saved automatically to `checkpoints/ppo_donkeykong/` every 100,000 steps.

In [4]:
if ALGO == "dqn":
    kwargs = dict(
        verbose=1, device="cpu", tensorboard_log="logs",
        buffer_size=100_000, learning_starts=10_000,
        batch_size=32, train_freq=4, target_update_interval=1_000,
    )
    if LEARNING_RATE is not None:
        kwargs["learning_rate"] = LEARNING_RATE
    model = DQN("CnnPolicy", vec_env, **kwargs)
elif ALGO == "ppo":
    kwargs = dict(
        verbose=1, device="cpu", tensorboard_log="logs",
        n_steps=128, batch_size=256, n_epochs=4, ent_coef=0.01, clip_range=0.1,
    )
    if LEARNING_RATE is not None:
        kwargs["learning_rate"] = LEARNING_RATE
    model = PPO("CnnPolicy", vec_env, **kwargs)

print(f"Algorithm     : {ALGO.upper()}")
print(f"Learning rate : {LEARNING_RATE}")
print(f"Device        : {model.device}")

Using cpu device
Wrapping the env in a VecTransposeImage.
Algorithm     : PPO
Learning rate : None
Device        : cpu


In [5]:
checkpoint_callback = CheckpointCallback(
    save_freq=max(CHECKPOINT_FREQ // n_envs, 1),
    save_path=checkpoint_dir,
    name_prefix="checkpoint",
    save_replay_buffer=False,
    save_vecnormalize=True,
)

model.learn(
    total_timesteps=TOTAL_TIMESTEPS,
    callback=checkpoint_callback,
    tb_log_name=run_id,
    log_interval=100,
    progress_bar=True,
    reset_num_timesteps=True,
)

final_path = os.path.join(checkpoint_dir, "final_model")
model.save(final_path)
print(f"\nTraining complete. Final model saved to: {final_path}.zip")

Logging to logs/ppo_donkeykong_1


Output()

KeyboardInterrupt: 